# External Validation Against CTRPv2 Cell Line Response Data

In [ ]:
from __future__ import annotations

import gc
import re
import warnings
from pathlib import Path
from datetime import datetime

import altair as alt
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer
from tensorflow import keras
from tqdm import tqdm

from cdrpy.datasets import Dataset
from cdrpy.feat.encoders import PandasEncoder
from cdrpy.mapper import BatchedResponseGenerator, FunctionAuxResponseGenerator
from cdrpy.metrics import tf_metrics

from screendl import model as screendl
from screendl.model import utils as model_utils
from screendl.pipelines.pdmc.utils import get_screenahead_split
from screendl.screenahead import PrincipalDrugSelector
from screendl.pipelines.basic.screendl import _similarity_matrix_to_targets

from utils.eval import auroc, select_best_therapy
from utils.plot import MODEL_COLORS, DEFAULT_BOXPLOT_CONFIG, configure_chart

In [ ]:
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
warnings.filterwarnings("ignore", message=".*set_learning_phase.*")

In [ ]:
store = Path("../../../datastore")

## Data Processing

In [ ]:
ctrp_resp_path = (
    store / "raw/CTRP/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.data.curves_post_qc.txt"
)
ctrp_curve_path = (
    store / "raw/CTRP/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.data.per_cpd_post_qc.txt"
)
ctrp_meta_path = (
    store / "raw/CTRP/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.meta.per_experiment.txt"
)
ctrp_t_meta_path = (
    store / "raw/CTRP/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.meta.per_cell_line.txt"
)
ctrp_d_meta_path = (
    store / "raw/CTRP/CTRPv2.0_2015_ctd2_ExpandedDataset/v20.meta.per_compound.txt"
)

ctrp_resp = pd.read_table(ctrp_resp_path)
ctrp_curve = pd.read_table(ctrp_curve_path)
ctrp_meta = pd.read_table(ctrp_meta_path)
ctrp_t_meta = pd.read_table(ctrp_t_meta_path)
ctrp_d_meta = pd.read_table(ctrp_d_meta_path)

In [ ]:
def sanitize_ccl_name(str_: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]", "", str_)

In [ ]:
dmap_t_meta = (
    pd.read_csv(store / "raw/DepMap/Model.csv")
    .filter(items=["ModelID", "SangerModelID", "CellLineName", "OncotreeLineage"])
    .dropna()
    .assign(ccl_name=lambda df: df["CellLineName"].apply(sanitize_ccl_name))
)
dmap_t_meta.head()

In [ ]:
# Drop cell lines without 1-1 mapping between name and clean name
counts = dmap_t_meta.groupby("ccl_name")["CellLineName"].nunique()
drop = counts[counts > 1].index.tolist()
dmap_t_meta = dmap_t_meta[~dmap_t_meta["ccl_name"].isin(drop)]
n_dmap_ccls = dmap_t_meta["CellLineName"].nunique()
print(f"Number of DepMap cell lines: {n_dmap_ccls}")

# confirm 1-1 mapping between master ccl ID and name in CDTP
assert ctrp_t_meta.groupby("master_ccl_id")["ccl_name"].nunique().nunique() == 1

ctrp_t_names = ctrp_t_meta[ctrp_t_meta["ccl_name"].isin(dmap_t_meta["ccl_name"])]["ccl_name"].tolist()
dmap_t_meta = dmap_t_meta[dmap_t_meta["ccl_name"].isin(ctrp_t_names)]
print(f"Number of mapped cell lines {len(ctrp_t_names)}")

ccl_name_to_id = dict(zip(dmap_t_meta["ccl_name"], dmap_t_meta["ModelID"]))

## Check Correlation Between DepMap and CellModelPassports Gene Expression

In [ ]:
sang_gexp_path = store / "inputs/CellModelPassports-GDSCv1v2/ScreenDL/FeatureGeneExpression.csv"
sang_gexp = pd.read_csv(sang_gexp_path, index_col=0)
sang_gexp.head()

In [ ]:
dmap_gexp_path = store / "raw/DepMap/OmicsExpressionProteinCodingGenesTPMLogp1.csv"
dmap_gexp = pd.read_csv(dmap_gexp_path, index_col=0)
dmap_gexp.columns = dmap_gexp.columns.map(lambda x: x.split(" ")[0])
dmap_gexp.head()

In [ ]:
dmap_gexp = pd.concat([sang_gexp.iloc[:0], dmap_gexp.filter(items=sang_gexp.columns)])
dmap_gexp.head()

In [ ]:
dmap_to_sang = dict(zip(dmap_t_meta["ModelID"], dmap_t_meta["SangerModelID"]))

dmap_gexp_common = dmap_gexp[dmap_gexp.index.isin(list(dmap_to_sang))].copy()
dmap_gexp_common.index = dmap_gexp_common.index.map(dmap_to_sang)
assert dmap_gexp_common.index.is_unique

In [ ]:
sang_gexp_common = sang_gexp[sang_gexp.index.isin(list(dmap_to_sang.values()))].copy()
assert sang_gexp_common.index.is_unique

In [ ]:
# check correspondence between expression in the two datasets
keep_genes = dmap_gexp_common.dropna(axis=1).columns

corrs = []
for t in sang_gexp_common.index.intersection(dmap_gexp_common.index):
    s1 = sang_gexp_common.loc[t, keep_genes]
    s2 = dmap_gexp_common.loc[t, keep_genes]
    corrs.append(stats.pearsonr(s1, s2)[0])

gexp_cross_dataset_corrs = pd.Series(corrs)
gexp_cross_dataset_corrs.describe()

In [ ]:
# impute missing feature values for the model
imputer = KNNImputer().fit(sang_gexp_common)
dmap_gexp_common.loc[:, :] = imputer.transform(dmap_gexp_common)
dmap_gexp_common.head()

## Get Drug Features

In [ ]:
# generate fingerprints and map drugs across datasets

from rdkit import Chem
from rdkit.Chem import AllChem

_morgan_gen = AllChem.GetMorganGenerator(radius=3, fpSize=512)


def smiles_to_morgan(smiles, gen=_morgan_gen):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return gen.GetFingerprintAsNumPy(mol)


fps = [smiles_to_morgan(x) for x in ctrp_d_meta["cpd_smiles"]]

In [ ]:
gdsc_mol_feat = pd.read_csv(
    store / "inputs/CellModelPassports-GDSCv1v2/ScreenDL/FeatureMorganFingerprints.csv",
    index_col=0,
)

In [ ]:
ctrp_fps = set(map(tuple, fps))
gdsc_fps = gdsc_mol_feat.apply(tuple, axis=1)
common_fps = gdsc_fps[gdsc_fps.isin(ctrp_fps)]
fp_to_drug = dict(zip(common_fps.values, common_fps.index))

In [ ]:
dmap_mol_feat = pd.DataFrame(common_fps.values, index=common_fps.index)

In [ ]:
common_d_ids = common_fps.index.tolist()

ctrp_d_meta_to_drug_id = (
    ctrp_d_meta[["master_cpd_id"]]
    .assign(drug_id=lambda df: [fp_to_drug.get(tuple(fp)) for fp in fps])
    .dropna()
)

# Compute IC50 for CTRPv2 responses

In [ ]:
from scipy.optimize import curve_fit


def log_logistic(c, emin, ec50, h):
    """4-param logistic with emax fixed at 1."""
    return emin + (1.0 - emin) / (1.0 + (c / ec50) ** h)


def log_logistic_constrained(c, ec50, h):
    """Both emin=0 and emax=1 fixed (matches GDS methodology)."""
    return 1.0 / (1.0 + (c / ec50) ** h)


def fit_ic50(conc, viability):
    """Fit dose-response and compute absolute IC50."""
    try:
        popt, _ = curve_fit(
            log_logistic_constrained,
            conc,
            viability,
            p0=[np.median(conc), 1.0],  # ec50, hill
            bounds=([1e-6, 0.1], [1e4, 10.0]),  # ec50, hill
            maxfev=5000,
        )
        ec50, h = popt
        # with both asymptotes fixed, ic50 = ec50
        return ec50, 0.0, ec50, h
    except RuntimeError:
        return np.nan, np.nan, np.nan, np.nan

In [ ]:
tmpdir = Path("./temp")
tmpdir.mkdir(exist_ok=True)

In [ ]:
keep_drugs = set(ctrp_d_meta_to_drug_id.master_cpd_id)
keep_experiments = set(
    dmap_t_meta.query("SangerModelID in @dmap_gexp_common.index")
    .merge(ctrp_t_meta, on="ccl_name")
    .merge(ctrp_meta[["master_ccl_id", "experiment_id"]], on="master_ccl_id")[
        "experiment_id"
    ]
)

ctrp_curve = ctrp_curve.query(
    "master_cpd_id in @keep_drugs and experiment_id in @keep_experiments"
)

In [ ]:
ctrp_ic50_path = tmpdir / "ctrp_ic50s.csv"

if not ctrp_ic50_path.exists():
    from tqdm import tqdm

    ctrp_ic50 = []
    grouped = ctrp_curve.sort_values(["experiment_id", "master_cpd_id"]).groupby(
        ["experiment_id", "master_cpd_id"]
    )
    for name, group in tqdm(grouped):
        exp, mol = name
        group_ = group.sort_values("cpd_conc_umol")
        vals = group_["cpd_avg_pv"].clip(0, 1).values
        conc = group_["cpd_conc_umol"].values

        if vals.min() > 0.8:
            ctrp_ic50.append(
                {
                    "experiment_id": exp,
                    "master_cpd_id": mol,
                    "ic50": np.inf,
                    "emin": np.nan,
                    "ec50": np.nan,
                    "h": np.nan,
                }
            )
            continue

        ctrp_ic50.append(
            {
                "experiment_id": exp,
                "master_cpd_id": mol,
                "max_conc": conc.max(),
                **dict(
                    zip(
                        ["ic50", "emin", "ec50", "h"],
                        fit_ic50(conc, vals),
                    )
                ),
            }
        )

    ctrp_ic50_df = pd.DataFrame(ctrp_ic50)
    ctrp_ic50_df.to_csv(ctrp_ic50_path, index=False)

else:
    ctrp_ic50_df = pd.read_csv(ctrp_ic50_path)

ctrp_ic50_df.head()

In [ ]:
INVALID = [np.nan, np.inf, -np.inf]

ctrp_ic50_df = ctrp_ic50_df.query("ic50 not in @INVALID")
ctrp_ic50_df.head()

In [ ]:
# cap ic50 at max_conc so we don't extrapolate outside of dose range
ctrp_ic50_df["ic50"] = ctrp_ic50_df.apply(
    lambda row: row["ic50"] if row["ic50"] <= row["max_conc"] else row["max_conc"],
    axis=1,
)

# keep drugs with <50% capped ratio
capped_ratio = ctrp_ic50_df.groupby("master_cpd_id").apply(
    lambda g: (g["max_conc"] == g["ic50"]).sum() / g.shape[0]
)
keep = capped_ratio[capped_ratio <= 0.5].index
ctrp_ic50_df = ctrp_ic50_df[ctrp_ic50_df["master_cpd_id"].isin(keep)]
ctrp_ic50_df.head()

In [ ]:
ctrp_obs = (
    ctrp_ic50_df.merge(
        ctrp_meta[["experiment_id", "master_ccl_id"]], on="experiment_id"
    )
    .merge(ctrp_t_meta[["master_ccl_id", "ccl_name"]], on="master_ccl_id")
    .merge(dmap_t_meta, on=["ccl_name"], how="left")
    .assign(ln_ic50=lambda df: np.log(df["ic50"]))
)
ctrp_obs.head()

In [ ]:
ctrp_resp_agg = (
    ctrp_resp.groupby(["experiment_id", "master_cpd_id"])
    .agg({"area_under_curve": "mean", "apparent_ec50_umol": "mean"})
    .reset_index()
)

ctrp_obs = ctrp_obs.merge(ctrp_resp_agg, on=["experiment_id", "master_cpd_id"])
ctrp_obs.head()

In [ ]:
# check correlation with CTRP's area_under_curve
intra_ds_corrs = (
    ctrp_obs.dropna(
        subset=["ln_ic50", "ec50", "area_under_curve", "apparent_ec50_umol"]
    )
    .query("ln_ic50 not in @INVALID")
    .query("area_under_curve not in @INVALID")
    .groupby("master_cpd_id")
    .apply(lambda g: stats.pearsonr(g["ln_ic50"], g["area_under_curve"])[0])
)

intra_ds_corrs.describe()

## Check Cross-Dataset Corrs (Response)

In [ ]:
gdsc_obs = pd.read_csv(store / "inputs/CellModelPassports-GDSCv1v2/LabelsLogIC50.csv")
gdsc_obs.head()

In [ ]:
t_ids_common = set(gdsc_obs["cell_id"]) & set(ctrp_obs["SangerModelID"])
gdsc_obs_common = gdsc_obs.query("drug_id in @common_d_ids").query(
    "cell_id in @t_ids_common"
)
ctrp_obs_common = ctrp_obs.merge(ctrp_d_meta_to_drug_id, on="master_cpd_id").query(
    "SangerModelID in @t_ids_common"
)

In [ ]:
ctrp_obs = ctrp_obs.merge(ctrp_d_meta_to_drug_id, on="master_cpd_id")

In [ ]:
obs_both = ctrp_obs_common.merge(
    gdsc_obs_common,
    left_on=["SangerModelID", "drug_id"],
    right_on=["cell_id", "drug_id"],
)

drug_concords = obs_both.groupby("drug_id").apply(
    lambda g: stats.pearsonr(g["ln_ic50"], g["label"])[0]
)

drug_concords.describe()

## Build Datasets and Fit Model

In [ ]:
dataset_dir = store / f"inputs/CellModelPassports-GDSCv1v2"

drug_encoders = screendl.load_drug_features(
    dataset_dir / "ScreenDL/FeatureMorganFingerprints.csv"
)

cell_encoders = screendl.load_cell_features(
    dataset_dir / "ScreenDL/FeatureGeneExpression.csv"
)

D_gdsc = Dataset.from_csv(
    dataset_dir / "LabelsLogIC50.csv",
    cell_encoders=cell_encoders,
    drug_encoders=drug_encoders,
    name=dataset_dir.name,
)

print(D_gdsc)

In [ ]:
from sklearn.preprocessing import StandardScaler

gexp_scaler = StandardScaler()
_ = gexp_scaler.fit(D_gdsc.cell_encoders["exp"].data.values)
D_gdsc.cell_encoders["exp"].data.loc[:, :] = gexp_scaler.transform(
    D_gdsc.cell_encoders["exp"].data.values
)

assert not False in set(
    dmap_gexp_common.columns == D_gdsc.cell_encoders["exp"].data.columns
)
dmap_gexp_common_norm = gexp_scaler.transform(dmap_gexp_common.values)

dmap_gexp_common_norm = pd.DataFrame(
    dmap_gexp_common_norm,
    index=dmap_gexp_common.index,
    columns=dmap_gexp_common.columns,
)

In [ ]:
ctrp_obs_final = (
    ctrp_obs.filter(items=["SangerModelID", "drug_id", "ln_ic50"])
    .rename(columns={"SangerModelID": "cell_id", "ln_ic50": "label"})
    .assign(id=range(len(ctrp_obs)))
    .query("cell_id in @dmap_gexp_common_norm.index")
    .query("label not in @INVALID")
)
ctrp_obs_final.head()

In [ ]:
D_ctrp = Dataset(
    obs=ctrp_obs_final,
    cell_encoders={"exp": PandasEncoder(dmap_gexp_common_norm)},
    drug_encoders={"mol": PandasEncoder(dmap_mol_feat[0].apply(pd.Series))}
)
print(D_ctrp)

In [ ]:
D_ctrp.drug_encoders["mol"].data = D_ctrp.drug_encoders["mol"].data.astype(np.int32)
D_ctrp.cell_encoders["exp"].data = D_ctrp.cell_encoders["exp"].data.astype(np.float32)
D_ctrp.obs["label"] = D_ctrp.obs["label"].astype(np.float32)

In [ ]:
D_gdsc.obs["label"] = D_gdsc.obs.groupby("drug_id")["label"].transform(stats.zscore)
D_ctrp.obs["label"] = D_ctrp.obs.groupby("drug_id")["label"].transform(stats.zscore)

In [ ]:
rmat = D_gdsc.obs.pivot_table(
    index="cell_id", columns="drug_id", values="label", aggfunc="mean"
)

drug_centered = rmat.sub(rmat.mean(axis=0), axis=1)
drug_sim = drug_centered.corr()

cell_centered = rmat.sub(rmat.mean(axis=1), axis=0)
cell_sim = cell_centered.T.corr()

drug_targets = _similarity_matrix_to_targets(
    drug_sim, n_components=16, prefix="drug_function", random_state=1771
)

cell_targets = _similarity_matrix_to_targets(
    cell_sim, n_components=16, prefix="cell_function", random_state=1771
)

In [ ]:
exp_dim = D_gdsc.cell_encoders["exp"].shape[-1]
mol_dim = D_gdsc.drug_encoders["mol"].shape[-1]

model = screendl.create_model(
    exp_dim,
    mol_dim,
    exp_norm_layer=None,
    cnv_norm_layer=None,
    exp_hidden_dims=[512, 256, 128, 64],
    mol_hidden_dims=[256, 128, 64],
    shared_hidden_dims=[128, 64],
    activation="leaky_relu",
    use_l2=False,
    use_noise=True,
    noise_stddev=0.3,
    interaction_mode="bilinear",
    bilinear_dim=64,
    include_bilinear_product=True,
    include_bilinear_score=False,
)

model = screendl.add_function_auxiliary_heads(
    model,
    drug_aux_dim=drug_targets.shape[1],
    cell_aux_dim=cell_targets.shape[1],
    drug_hidden_dims=None,
    cell_hidden_dims=None,
    activation="leaky_relu",
    use_l2=False,
    l2_factor=False,
)

In [ ]:
opt = keras.optimizers.Adam(1e-4, weight_decay=1e-4)

model = screendl.train_model(
    model,
    opt,
    D_gdsc,
    None,
    batch_size=256,
    epochs=15,
    save_dir=None,
    log_dir=None,
    early_stopping=False,
    tensorboard=False,
    drug_function_targets=drug_targets,
    cell_function_targets=cell_targets,
    drug_function_loss_weight=0.01,
    cell_function_loss_weight=0.01,
)

model = screendl.get_response_model(model)

In [ ]:
gen = BatchedResponseGenerator(D_ctrp, batch_size=256)
preds = D_ctrp.obs.rename(columns={"label": "y_true"}).copy()
preds["y_pred"] = model.predict(gen.flow_from_dataset(D_ctrp)).flatten()
preds["y_pred"] = preds.groupby("drug_id")["y_pred"].transform(stats.zscore)
preds.head()

In [ ]:
corrs = (
    preds.groupby("drug_id")
    .apply(
        lambda g: pd.Series(
            {
                "pcc": (
                    stats.pearsonr(g["y_true"], g["y_pred"])[0]
                    if len(g) > 5
                    else np.nan
                ),
                "scc": (
                    stats.spearmanr(g["y_true"], g["y_pred"])[0]
                    if len(g) > 5
                    else np.nan
                ),
                "n_cells": g["cell_id"].nunique(),
            }
        )
    )
    .sort_values("pcc")
)

corrs.describe()

In [ ]:
corrs_ood = (
    preds.query("cell_id not in @D_gdsc.cell_ids").groupby("drug_id")
    .apply(
        lambda g: pd.Series(
            {
                "pcc": (
                    stats.pearsonr(g["y_true"], g["y_pred"])[0]
                    if len(g) > 5
                    else np.nan
                ),
                "scc": (
                    stats.spearmanr(g["y_true"], g["y_pred"])[0]
                    if len(g) > 5
                    else np.nan
                ),
                "n_cells": g["cell_id"].nunique(),
            }
        )
    )
    .sort_values("pcc")
)

corrs_ood.describe()

In [ ]:
AUX_ENABLED = True

# FT / transfer settings
XFER_EPOCHS = 30
XFER_BATCH_SIZE = 32  # change if your original generator used another batch size
XFER_LR = 1e-4
XFER_WEIGHT_DECAY = 1e-4

XFER_FROZEN_LAYER_NAMES = (
    "exp_mlp_1",
    "mol_mlp_1",
    "mol_mlp_2",
)
XFER_FROZEN_LAYER_PREFIXES = ()

# AUX settings
AUX_DRUG_N_COMPONENTS = 16
AUX_CELL_N_COMPONENTS = 16
AUX_SEED = 1441
AUX_RESPONSE_LOSS_WEIGHT = 1.0
AUX_DRUG_LOSS_WEIGHT = 0.001
AUX_CELL_LOSS_WEIGHT = 0.001
AUX_DRUG_HIDDEN_DIMS = None
AUX_CELL_HIDDEN_DIMS = None
AUX_ACTIVATION = "relu"
AUX_USE_L2 = False
AUX_L2_FACTOR = 0.01

# ScreenAhead settings
SA_EPOCHS = 20
SA_BATCH_SIZE = 32  # change if your original generator used another batch size
SA_LR = 1e-4
SA_WEIGHT_DECAY = None
SA_N_DRUGS = 20
SA_SELECTOR_SEED = 1771

SA_FROZEN_LAYER_NAMES = ()
SA_FROZEN_LAYER_PREFIXES = ("exp", "mol")

In [ ]:
def _similarity_matrix_to_targets(
    sim_df: pd.DataFrame,
    *,
    n_components: int,
    prefix: str,
    fill_value: float = 0.0,
    seed: int = 1441,
) -> pd.DataFrame:
    common = sim_df.index.intersection(sim_df.columns)
    sim_df = sim_df.loc[common, common]

    columns = [f"{prefix}_{i}" for i in range(n_components)]
    if sim_df.empty:
        return pd.DataFrame(columns=columns, dtype="float32")

    x = sim_df.fillna(fill_value).to_numpy(dtype="float32")
    k = min(n_components, x.shape[0], x.shape[1])

    z = PCA(n_components=k, random_state=seed).fit_transform(x).astype("float32")
    if k < n_components:
        z_padded = pd.DataFrame(
            0.0,
            index=sim_df.index,
            columns=columns,
            dtype="float32",
        )
        z_padded.iloc[:, :k] = z
        return z_padded

    return pd.DataFrame(z, index=sim_df.index, columns=columns)


def make_fold_function_aux_targets(
    D_train,
    *,
    drug_n_components: int,
    cell_n_components: int,
    cell_col: str = "cell_id",
    drug_col: str = "drug_id",
    label_col: str = "label",
    seed: int = 1441,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build fold-specific drug/tumor functional AUX targets from D_rest only."""
    response_mat = D_train.obs.pivot_table(
        index=cell_col,
        columns=drug_col,
        values=label_col,
        aggfunc="mean",
    )

    drug_centered = response_mat.sub(response_mat.mean(axis=0), axis=1)
    drug_sim = drug_centered.corr()

    tumor_centered = response_mat.sub(response_mat.mean(axis=1), axis=0)
    tumor_sim = tumor_centered.T.corr()

    drug_targets = _similarity_matrix_to_targets(
        drug_sim,
        n_components=drug_n_components,
        prefix="drug_function",
        seed=seed,
    )
    cell_targets = _similarity_matrix_to_targets(
        tumor_sim,
        n_components=cell_n_components,
        prefix="cell_function",
        seed=seed,
    )

    return drug_targets, cell_targets


def fit_aux_transfer_model(M_base, W_base, D_train):
    """Fit response + functional AUX transfer model, then return response model."""
    drug_targets, cell_targets = make_fold_function_aux_targets(
        D_train,
        drug_n_components=AUX_DRUG_N_COMPONENTS,
        cell_n_components=AUX_CELL_N_COMPONENTS,
        seed=AUX_SEED,
    )

    M_aux_base = None
    M_aux = None
    gen = None
    seq = None

    try:
        # Critical: do not attach AUX heads to the long-lived base model.
        M_aux_base = model_utils.clone_model_from_weights(M_base, W_base)
        M_aux_base.trainable = True

        M_aux = screendl.add_function_auxiliary_heads(
            M_aux_base,
            drug_aux_dim=AUX_DRUG_N_COMPONENTS,
            cell_aux_dim=AUX_CELL_N_COMPONENTS,
            drug_hidden_dims=AUX_DRUG_HIDDEN_DIMS,
            cell_hidden_dims=AUX_CELL_HIDDEN_DIMS,
            activation=AUX_ACTIVATION,
            use_l2=AUX_USE_L2,
            l2_factor=AUX_L2_FACTOR,
        )

        model_utils.freeze_layers(
            M_aux,
            names=XFER_FROZEN_LAYER_NAMES,
            prefixes=XFER_FROZEN_LAYER_PREFIXES,
        )

        M_aux.compile(
            optimizer=keras.optimizers.Adam(
                learning_rate=XFER_LR,
                weight_decay=XFER_WEIGHT_DECAY,
            ),
            loss={
                "response": "mean_squared_error",
                "drug_function": "mean_squared_error",
                "cell_function": "mean_squared_error",
            },
            loss_weights={
                "response": AUX_RESPONSE_LOSS_WEIGHT,
                "drug_function": AUX_DRUG_LOSS_WEIGHT,
                "cell_function": AUX_CELL_LOSS_WEIGHT,
            },
            jit_compile=False,
        )

        gen = FunctionAuxResponseGenerator(D_train, XFER_BATCH_SIZE)
        seq = gen.flow_from_dataset(
            D_train,
            drug_function_targets=drug_targets,
            cell_function_targets=cell_targets,
            shuffle=True,
            seed=AUX_SEED,
        )

        M_aux.fit(seq, epochs=XFER_EPOCHS, verbose=0)

        M_tune = screendl.get_response_model(M_aux)
        W_tune = M_tune.get_weights()
        return M_tune, W_tune

    finally:
        del seq
        del gen
        del M_aux_base
        gc.collect()


def fit_response_only_transfer_model(M_base, W_base, D_train):
    """Fallback matching the old response-only fine-tuning path."""
    M_tune = model_utils.configure_transfer_model(
        M_base,
        initial_weights=W_base,
        frozen_layer_prefixes=XFER_FROZEN_LAYER_PREFIXES,
        frozen_layer_names=XFER_FROZEN_LAYER_NAMES,
    )

    M_tune.compile(
        keras.optimizers.Adam(
            learning_rate=XFER_LR,
            weight_decay=XFER_WEIGHT_DECAY,
        ),
        loss="mse",
    )
    M_tune.fit(
        gen.flow_from_dataset(D_train, shuffle=True),
        epochs=XFER_EPOCHS,
        verbose=0,
    )

    W_tune = M_tune.get_weights()
    return M_tune, W_tune


def fit_screenahead_model(M_tune, W_tune, D_screen):
    """Fit ScreenAhead from the tuned response model."""
    M_screen = model_utils.configure_screenahead_model(
        M_tune,
        initial_weights=W_tune,
        frozen_layer_prefixes=SA_FROZEN_LAYER_PREFIXES,
        frozen_layer_names=SA_FROZEN_LAYER_NAMES,
        training=False,
    )

    M_screen.compile(
        keras.optimizers.Adam(
            learning_rate=SA_LR,
            weight_decay=SA_WEIGHT_DECAY,
        ),
        loss="mse",
    )
    M_screen.fit(
        gen.flow_from_dataset(D_screen, shuffle=True),
        epochs=SA_EPOCHS,
        verbose=0,
    )

    return M_screen, M_screen.get_weights()


def predict_df(M, D, *, model_name: str, was_screened=False):
    pred = D.obs.rename(columns={"label": "y_true"}).copy()
    pred["y_pred"] = M.predict(gen.flow_from_dataset(D), verbose=0).flatten()

    if isinstance(was_screened, bool):
        pred["was_screened"] = was_screened
    else:
        pred["was_screened"] = pred["drug_id"].isin(set(was_screened))

    pred["model"] = model_name
    return pred

In [ ]:
M_base = model
W_base = model.get_weights()

results = []

test_t_ids = sorted(list(set(D_ctrp.cell_ids) - set(D_gdsc.cell_ids)))

for fold_i, t_id in enumerate(tqdm(test_t_ids)):
    D_rest = D_ctrp.select_cells([x for x in test_t_ids if x != t_id])
    D_this = D_ctrp.select_cells([t_id])

    if D_this.n_drugs <= SA_N_DRUGS:
        continue

    # Base / pretrained prediction.
    # Use a fresh clone so nothing later can accidentally mutate M_base.
    M_fold_base = model_utils.clone_model_from_weights(M_base, W_base)

    B_preds = predict_df(
        M_fold_base,
        D_this,
        model_name="ScreenDL-PT",
        was_screened=False,
    )
    B_preds["fold_i"] = fold_i
    B_preds["heldout_cell_id"] = t_id
    results.append(B_preds)

    # Transfer / fine-tuning.
    if AUX_ENABLED:
        M_tune, W_tune = fit_aux_transfer_model(M_base, W_base, D_rest)
    else:
        M_tune, W_tune = fit_response_only_transfer_model(M_base, W_base, D_rest)

    T_preds = predict_df(
        M_tune,
        D_this,
        model_name="ScreenDL-FT",
        was_screened=False,
    )
    T_preds["fold_i"] = fold_i
    T_preds["heldout_cell_id"] = t_id
    results.append(T_preds)

    # ScreenAhead split uses the N-1 background tumor panel selector,
    # then screens only drugs from the held-out tumor.
    D_screen, _ = get_screenahead_split(
        D_this,
        drug_selector=PrincipalDrugSelector(D_rest, seed=SA_SELECTOR_SEED),
        num_drugs=SA_N_DRUGS,
        exclude_drugs=None,
    )

    M_screen, W_screen = fit_screenahead_model(M_tune, W_tune, D_screen)

    S_preds = predict_df(
        M_screen,
        D_this,
        model_name="ScreenDL-SA",
        was_screened=D_screen.drug_ids,
    )
    S_preds["fold_i"] = fold_i
    S_preds["heldout_cell_id"] = t_id
    results.append(S_preds)

    # Per-fold cleanup.
    model_utils.clear_compiled_model(M_fold_base)
    model_utils.clear_compiled_model(M_tune)
    model_utils.clear_compiled_model(M_screen)

    del M_fold_base, M_tune, M_screen, W_tune, W_screen
    gc.collect()

results_df = pd.concat(results, ignore_index=True)
results_df.head()

In [ ]:
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def norm_mse(y_true, y_pred):
    return np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)


def norm_mae(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true - np.mean(y_true)))


def compute_group_metrics(
    group: pd.DataFrame, bin_col: str = "y_true_bin_drug"
) -> pd.Series:
    if len(group) < 10:
        return pd.Series(
            {
                "pcc": np.nan,
                "scc": np.nan,
                "auc": np.nan,
                "mse": np.nan,
                "mae": np.nan,
                "norm_mse": np.nan,
                "norm_mae": np.nan,
                "n": group.shape[0],
            }
        )
    return pd.Series(
        {
            "pcc": stats.pearsonr(group["y_true"], group["y_pred"])[0],
            "scc": stats.spearmanr(group["y_true"], group["y_pred"])[0],
            "auc": auroc(group, bin_col, "y_pred"),
            "mse": mse(group["y_true"], group["y_pred"]),
            "mae": mae(group["y_true"], group["y_pred"]),
            "norm_mse": norm_mse(group["y_true"], group["y_pred"]),
            "norm_mae": norm_mae(group["y_true"], group["y_pred"]),
            "n": group.shape[0],
        }
    )

In [ ]:
assign_bin = lambda g: (g <= g.quantile(0.3)).astype(int)

y_true_df = (
    D_ctrp.obs.copy()
    .assign(
        y_true_bin_drug=lambda df: df.groupby("drug_id")["label"].transform(assign_bin)
    )
    .assign(
        y_true_bin_cell=lambda df: df.groupby("cell_id")["label"].transform(assign_bin)
    )
    .filter(items=["cell_id", "drug_id", "y_true_bin_drug", "y_true_bin_cell"])
)

results_df = results_df.merge(y_true_df, on=["cell_id", "drug_id"])

In [ ]:
outdir = Path("./temp/ctrp-val")
outdir.mkdir(exist_ok=True)

ts = datetime.now().strftime("%Y%m%d-%H%M%S")
results_df.to_csv(outdir / f"results-{ts}.csv", index=False)

In [ ]:
MODELS = ["ScreenDL-PT", "ScreenDL-FT", "ScreenDL-SA"]

model_to_color = {k: v for k, v in MODEL_COLORS.items() if k in MODELS}
MODEL_COLOR_SCALE = alt.Scale(
    domain=list(model_to_color.keys()), range=list(model_to_color.values())
)

In [ ]:
d_metrics = (
    results_df.query("cell_id not in @D_gdsc.cell_ids")
    .groupby(["model", "drug_id"])
    .apply(compute_group_metrics)
    .query("n >= 10")
)

d_metrics.groupby("model").median().T[["ScreenDL-PT", "ScreenDL-FT", "ScreenDL-SA"]]

In [ ]:
RR_d = (
    results_df
    .query("cell_id not in @D_gdsc.cell_ids")
    .groupby(["model", "cell_id"])
    .apply(select_best_therapy)
    .groupby("model")["y_true_bin_drug"]
    .mean()
    .to_frame(name="RR")
    .T[["ScreenDL-PT", "ScreenDL-FT", "ScreenDL-SA"]]
)
RR_d

In [ ]:
pcc_chart = (
    alt.Chart(d_metrics.reset_index(), width=50 * len(MODELS), height=300)
    .mark_boxplot(**{**DEFAULT_BOXPLOT_CONFIG, "size": 40})
    .encode(
        alt.X("model:N")
        .axis(labelAngle=-45, labelPadding=5)
        .scale(paddingOuter=0.1)
        .sort(MODELS)
        .title(None),
        alt.Y("pcc:Q")
        .axis(titlePadding=10, tickCount=6, grid=False)
        .scale(domain=[-1, 1])
        .title("Pearson Correlation"),
        alt.Color("model:N", scale=MODEL_COLOR_SCALE).legend(None),
    )
)

In [ ]:
bars = (
    alt.Chart()
    .mark_bar(stroke="black", size=40, strokeWidth=1, fillOpacity=1, strokeOpacity=1)
    .encode(
        alt.Y("median(auc):Q")
        .axis(grid=False, tickCount=5, domainColor="black", titlePadding=10)
        .scale(domain=(0.0, 1.0))
        .title("auROC"),
        alt.X("model:N")
        .axis(domainColor="black", labelAngle=-45)
        .scale(domain=MODELS, paddingOuter=0.15)
        .title(None),
        alt.Color("model:N", scale=MODEL_COLOR_SCALE).legend(None),
    )
    .properties(width=50 * len(MODELS), height=300)
)

error_bars = (
    alt.Chart()
    .mark_errorbar(
        extent="iqr", ticks=alt.MarkConfig(size=5, color="black", strokeWidth=1)
    )
    .encode(alt.X("model:N"), alt.Y("auc:Q").title(None))
)

rule = (
    alt.Chart(pd.DataFrame({"y": [0.5]}))
    .mark_rule(stroke="black", strokeDash=[4, 3], strokeWidth=1.25)
    .encode(y="y:Q")
)

auroc_chart = alt.layer(bars, error_bars, rule, data=d_metrics.reset_index())

In [ ]:
rr_chart = (
    alt.Chart(RR_d.melt(ignore_index=False))
    .mark_bar(stroke="black", size=40, strokeWidth=1, fillOpacity=1, strokeOpacity=1)
    .encode(
        alt.Y("value:Q")
        .axis(grid=False, tickCount=5, domainColor="black", titlePadding=10)
        .scale(domain=(0.0, 1.0))
        .title("Response Rate (%)"),
        alt.X("model:N")
        .axis(domainColor="black", labelAngle=-45)
        .scale(domain=MODELS, paddingOuter=0.15)
        .title(None),
        alt.Color("model:N", scale=MODEL_COLOR_SCALE).legend(None),
    )
    .properties(width=50 * len(MODELS), height=300)
)

In [ ]:
panel = alt.hconcat(pcc_chart, auroc_chart, rr_chart, spacing=80)
configure_chart(panel)